<a href="https://colab.research.google.com/github/geopayme/AstroPhysics/blob/main/ewpd4lhc_wilson_ray_colab_auto_FIXED_v2_sanitize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EWPD4LHC → Wilson-Ray Inputs (Flavor-Universal Default) + χ²⊥ Test (Fully Automatic YAML Extraction)

This notebook:

1. Clones `ewpd4lhc/ewpd4lhc` and runs the default flavor-universal build.
2. Loads the produced YAML output.
3. **Automatically locates** within the YAML:
   - coefficient/operator name list `coeff_names`,
   - response/Jacobian matrix `A` (observables × coefficients),
   - observable covariance `V` or precision `Vinv` (observables × observables),
   by scanning all nested keypaths and selecting a **shape-consistent triple**.
4. Constructs the coefficient-space Fisher matrix `F = Aᵀ V⁻¹ A` and `SigmaC = pinv(F)`.
5. Computes χ²⊥ after you paste `v_ray` in the discovered coefficient ordering.

If the YAML does not contain a covariance/precision matrix (rare), the notebook stops with a clear diagnostic
showing the top candidate matrices and string lists.

## 0) Environment

In [ ]:
import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())

## 1) Clone repo

In [ ]:
!git clone https://github.com/ewpd4lhc/ewpd4lhc.git
%cd ewpd4lhc
!ls -la

## 2) Dependencies

In [ ]:
!pip -q install numpy pyyaml scipy pandas

## 3) Run default build

In [ ]:
!chmod +x ewpd4lhc.py
!./ewpd4lhc.py
!ls -lh

## 4) Load YAML output

In [ ]:
import yaml
from pathlib import Path

candidates = ["ewpd_out.yml", "ewpd_out.yaml", "out.yml", "out.yaml"]
yml_path = None
for c in candidates:
    p = Path(c)
    if p.exists():
        yml_path = p
        break
if yml_path is None:
    ymls = sorted(list(Path(".").glob("*.yml")) + list(Path(".").glob("*.yaml")))
    if not ymls:
        raise FileNotFoundError("No .yml/.yaml output found after running ewpd4lhc.py.")
    yml_path = ymls[0]

print("Using YAML:", yml_path)

with open(yml_path, "r") as f:
    Y = yaml.safe_load(f)

print("Top-level type:", type(Y).__name__)
if isinstance(Y, dict):
    print("Top-level keys (first 80):", list(Y.keys())[:80])

In [ ]:
# --- Optional: sanitize YAML content (replace NaN/Inf) so downstream converters don't crash ---
import math

def sanitize_numbers(obj):
    if isinstance(obj, dict):
        return {k: sanitize_numbers(v) for k,v in obj.items()}
    if isinstance(obj, list):
        return [sanitize_numbers(v) for v in obj]
    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            return None
    return obj

Y_sanitized = sanitize_numbers(Y)

# Write sanitized YAML next to original
san_path = yml_path.with_name(yml_path.stem + "_sanitized" + yml_path.suffix)
with open(san_path, "w") as f:
    yaml.safe_dump(Y_sanitized, f, sort_keys=False)

print("Wrote sanitized YAML:", san_path)


## 5) Fully automatic extraction of `coeff_names`, `A`, `V`/`Vinv`

Algorithm:

- Scan all nested YAML keypaths.
- Collect candidates:
  - string lists (potential coefficient/operator names),
  - numeric matrices (potential A, V, Vinv).
- Select a consistent triple by shape:
  - `coeff_names`: length N
  - `A`: shape (M, N)
  - `V` or `Vinv`: shape (M, M)

If multiple triples exist, prefer the one with the largest `M×N`.

In [ ]:
import numpy as np
import re

# -------------------------------------------------------------------
# Robust YAML structure discovery for (coeff_names, A, V/Vinv)
# -------------------------------------------------------------------
# Accepts matrices stored as:
#   1) list-of-lists (numeric or numeric strings)
#   2) dict with {"rows":M,"cols":N,"data":[...]} (flattened)
#   3) dict with {"shape":[M,N],"data":[...]} (flattened)
#   4) dict with {"__ndarray__":[...], "shape":[M,N]} (numpy-ish)
#
# Outputs:
#   coeff_names : list[str]
#   A          : np.ndarray shape (M,N)
#   V_or_Vinv  : np.ndarray shape (M,M)
#   Vinv       : np.ndarray shape (M,M) (pseudo-inverse of V_or_Vinv)
#   PATH_*     : selected YAML paths (tuples of keys)
# -------------------------------------------------------------------

def iter_paths(obj, prefix=()):
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield from iter_paths(v, prefix + (str(k),))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from iter_paths(v, prefix + (f"[{i}]",))
    else:
        yield prefix, obj

def get_by_path(root, path):
    obj = root
    for k in path:
        if k.startswith("[") and k.endswith("]"):
            obj = obj[int(k[1:-1])]
        else:
            obj = obj[k]
    return obj

_num_re = re.compile(r"^[\+\-]?(?:\d+\.?\d*|\.\d+)(?:[eE][\+\-]?\d+)?$")

def to_float(x):
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    if isinstance(x, str):
        s = x.strip()
        if _num_re.match(s):
            return float(s)
    raise TypeError(f"Non-numeric scalar encountered: {type(x)}")

def list_is_str_list(L, min_len=5):
    return isinstance(L, list) and len(L) >= min_len and all(isinstance(x, str) for x in L)

def try_decode_matrix(val):
    # list-of-lists
    if isinstance(val, list) and val and all(isinstance(r, list) for r in val):
        try:
            M = len(val)
            N = len(val[0])
            if N == 0 or not all(len(r) == N for r in val):
                return None
            arr = np.array([[to_float(x) for x in r] for r in val], dtype=float)
            return arr
        except Exception:
            return None

    # dict encodings
    if isinstance(val, dict):
        # numpy-ish
        if "__ndarray__" in val and "shape" in val:
            try:
                shp = tuple(val["shape"])
                data = val["__ndarray__"]
                arr = np.array([to_float(x) for x in data], dtype=float).reshape(shp)
                return arr if arr.ndim == 2 else None
            except Exception:
                pass

        # {shape:[M,N], data:[...]}
        if "shape" in val and "data" in val:
            try:
                shp = tuple(val["shape"])
                data = val["data"]
                arr = np.array([to_float(x) for x in data], dtype=float).reshape(shp)
                return arr if arr.ndim == 2 else None
            except Exception:
                pass

        # {rows:M, cols:N, data:[...]}
        if "rows" in val and "cols" in val and "data" in val:
            try:
                M = int(val["rows"]); N = int(val["cols"])
                data = val["data"]
                arr = np.array([to_float(x) for x in data], dtype=float).reshape((M, N))
                return arr
            except Exception:
                pass

    return None

# Collect candidates
str_lists = []   # (path, N, sample)
matrices = []    # (path, (M,N), arr)

for path, val in iter_paths(Y):
    if list_is_str_list(val):
        str_lists.append((path, len(val), val[:5]))
    else:
        arr = try_decode_matrix(val)
        if arr is not None:
            matrices.append((path, arr.shape, arr))

# Find best shape-consistent triple
triples = []
for p_names, N, sample in str_lists:
    for pA, (M, N2), Aarr in matrices:
        if N2 != N:
            continue
        for pV, (MV, NV), Varr in matrices:
            if (MV, NV) != (M, M):
                continue
            # Prefer largest A (more constraints) and shallowest nesting
            score = (M * N, -len(p_names) - len(pA) - len(pV))
            triples.append((score, p_names, pA, pV, (M, N)))

if not triples:
    print("FAILED: No shape-consistent (coeff_names, A, V) triple found in YAML.")
    print("\nTop string-list candidates:")
    for p,n,s in sorted(str_lists, key=lambda x: -x[1])[:20]:
        print("  ", " / ".join(p), "len=", n, "sample=", s)
    print("\nTop matrix candidates:")
    for p,sh,_ in sorted(matrices, key=lambda x: -(x[1][0]*x[1][1]))[:20]:
        print("  ", " / ".join(p), "shape=", sh)
    raise ValueError("YAML does not contain decodeable (coeff_names, A, V/Vinv) structures.")

triples.sort(reverse=True, key=lambda x: x[0])
_, PATH_COEFF_NAMES, PATH_A, PATH_VORVINV, (M, N) = triples[0]

# Extract the selected objects
coeff_names = list(get_by_path(Y, PATH_COEFF_NAMES))
A = next(arr for p,sh,arr in matrices if p == PATH_A)
V_or_Vinv = next(arr for p,sh,arr in matrices if p == PATH_VORVINV)

# Always work with a stable pseudo-inverse for downstream quadratic forms
Vinv = np.linalg.pinv(V_or_Vinv)

print("Selected triple:")
print("  coeff_names path:", " / ".join(PATH_COEFF_NAMES), "(N=", len(coeff_names), ")")
print("  A path:", " / ".join(PATH_A), "shape=", tuple(A.shape))
print("  V/Vinv path:", " / ".join(PATH_VORVINV), "shape=", tuple(V_or_Vinv.shape))
print("cond(V_or_Vinv) ~", float(np.linalg.cond(V_or_Vinv)))


## 6) Build Fisher matrix and rank

\[
F = A^{T} V^{-1} A, \qquad \Sigma_C = F^{+}.
\]

In [ ]:
F = A.T @ Vinv @ A
SigmaC = np.linalg.pinv(F)

svals = np.linalg.svd(F, compute_uv=False)
tol = max(F.shape) * np.max(svals) * 1e-12
rankF = int(np.sum(svals > tol))

print("A shape:", A.shape)
print("F shape:", F.shape)
print("rank(F):", rankF, "out of", F.shape[0])
print("Smallest singular values (last 10):", svals[-10:])

## 7) Coefficient ordering and ray vector `v_ray`

Paste your ray vector `v_ray` in the displayed ordering. Scale is irrelevant.

In [ ]:
import pandas as pd
display(pd.DataFrame({"i": range(len(coeff_names)), "coef": coeff_names}).head(200))
print("Total coefficients:", len(coeff_names))

v_ray = None  # paste list/array here, length must equal len(coeff_names)

## 8) χ²⊥ computation

Uses Fisher quadratic form \(F\). Effective dof: \(\nu_{\mathrm{eff}}=\mathrm{rank}(F)-1\).

In [ ]:
def chi2_perp_from_F(C_hat, F, v_ray):
    C = np.asarray(C_hat, dtype=float).reshape(-1, 1)
    v = np.asarray(v_ray, dtype=float).reshape(-1, 1)
    num = float(v.T @ F @ C)
    den = float(v.T @ F @ v)
    if den <= 0:
        raise ValueError("Non-positive v^T F v. v may lie in a null direction or F ill-conditioned.")
    chi2 = float(C.T @ F @ C - (num*num)/den)
    return chi2, num, den

# Default: SM-centered (C_hat = 0) unless you later load a best-fit coefficient vector
C_hat = np.zeros((len(coeff_names), 1), dtype=float)

if v_ray is None:
    print("Set v_ray in the previous cell and re-run.")
else:
    chi2, num, den = chi2_perp_from_F(C_hat, F, v_ray)
    nu_eff = max(rankF - 1, 0)
    print("chi2_perp =", chi2)
    print("rank(F)   =", rankF)
    print("nu_eff    =", nu_eff)
    print("v^T F C   =", num)
    print("v^T F v   =", den)

## 9) Save arrays for download/archival

In [ ]:
from pathlib import Path
np.save("coeff_names.npy", np.array(coeff_names, dtype=object))
np.save("A.npy", A)
np.save("Vinv.npy", Vinv)
np.save("F.npy", F)
np.save("SigmaC_pinv.npy", SigmaC)
np.save("C_hat.npy", C_hat)

print("Saved .npy files in:", Path(".").resolve())
!ls -lh *.npy

## 10) Download (Colab)

In [ ]:
# from google.colab import files
# for fn in ["coeff_names.npy","A.npy","Vinv.npy","F.npy","SigmaC_pinv.npy","C_hat.npy"]:
#     files.download(fn)